In [2]:
import torch
# torchvision is not used in this notebook, so it can be removed
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

%config InlineBackend.figure_format = 'svg'

In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

In [7]:
!wget -nc --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -n ml-latest-small.zip

ratings_data = pd.read_csv('C:\\Users\\devya\\OneDrive\\Desktop\\AML\\exp_5\\ml-latest-small\\ml-latest-small\\ratings.csv')
movie_names_data = pd.read_csv('C:\\Users\\devya\\OneDrive\\Desktop\\AML\\exp_5\\ml-latest-small\\ml-latest-small\\movies.csv')

'wget' is not recognized as an internal or external command,
operable program or batch file.
'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
n_movies = len(movie_names_data)
n_user = len(ratings_data['userId'].unique())

In [9]:
ratings_data = pd.merge(ratings_data, movie_names_data, on='movieId', how='inner')

In [10]:

ratings_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [11]:

from sklearn.preprocessing import LabelEncoder
import random
Y = ratings_data.rating
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
X = np.array([user_enc.fit_transform(ratings_data.userId),
              movie_enc.fit_transform(ratings_data.title)]).T

In [12]:

user_enc.classes_[4], movie_enc.classes_[8871]

(np.int64(5), 'Toy Story (1995)')

In [13]:

for x, y in zip(X[:10], Y[:10]):
    print(list(x), y)

[np.int64(0), np.int64(8871)] 4.0
[np.int64(0), np.int64(3661)] 4.0
[np.int64(0), np.int64(3845)] 4.0
[np.int64(0), np.int64(7523)] 5.0
[np.int64(0), np.int64(9119)] 5.0
[np.int64(0), np.int64(3252)] 3.0
[np.int64(0), np.int64(1284)] 5.0
[np.int64(0), np.int64(1337)] 4.0
[np.int64(0), np.int64(7180)] 5.0
[np.int64(0), np.int64(1535)] 5.0


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [15]:

num_users = len(X)
num_movies = len(X)

In [16]:
from keras.layers import Input, Embedding, Flatten, Dot, Dense, Activation, Dropout
from keras.models import Model

def build_model():
    movie_input = Input(shape=[1], name="Book-Input")
    movie_embedding = Embedding(n_movies+1, 15, name="Book-Embedding")(movie_input)
    movie_vec = Flatten(name="Flatten-Books")(movie_embedding)

    user_input = Input(shape=[1], name="User-Input")
    user_embedding = Embedding(n_user+1, 15, name="User-Embedding")(user_input)
    user_vec = Flatten(name="Flatten-Users")(user_embedding)

    prod = Dot(name="Dot-Product", axes=1)([user_vec, movie_vec])

    prod = Dense(32)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)

    prod = Dense(16)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)
    prod = Dense(1)(prod)


    model = Model([user_input, movie_input], prod)
    model.compile('adam', 'mean_squared_error', metrics=['accuracy'])

    return model


model = build_model()

In [17]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./checkpoint.weights.h5',
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1)

history = model.fit([X_train[:, 0], X_train[:, 1]], Y_train,
            epochs=15,
            verbose=1,
            batch_size=64,
            validation_data=([X_test[:, 0], X_test[:, 1]], Y_test),
            callbacks=[model_checkpoint_callback])

Epoch 1/15
1246/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0219 - loss: 5.0442
Epoch 1: val_loss improved from inf to 1.20711, saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0219 - loss: 5.0190 - val_accuracy: 0.0280 - val_loss: 1.2071
Epoch 2/15
1257/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0270 - loss: 1.8583
Epoch 2: val_loss improved from 1.20711 to 1.17291, saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.0270 - loss: 1.8580 - val_accuracy: 0.0280 - val_loss: 1.1729
Epoch 3/15
1260/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0263 - loss: 1.5414
Epoch 3: val_loss improved from 1.17291 to 1.09359, saving model to ./checkpoint.weights.h5
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.0263 - loss: 1.5413 - val_accuracy: 0.0280 - val_loss: 1.0936
Epoch 4/15
1254/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0285 - loss: 1.1965
Epoch 4: val_loss 

In [18]:

X_test[:5], Y_test[:5]

(array([[ 275, 4337],
        [ 598, 7425],
        [ 482,  334],
        [ 201, 3548],
        [ 273, 3540]]),
 41008    5.0
 94274    2.5
 77380    2.5
 29744    3.0
 40462    4.0
 Name: rating, dtype: float64)

In [19]:

predictions = model.predict([X_test[:5, 0], X_test[:5, 1]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step


In [20]:

print(predictions,"\n\n", Y_test[:5].values)

[[3.546557 ]
 [3.8471465]
 [2.8240488]
 [3.1036596]
 [3.5052364]] 

 [5.  2.5 2.5 3.  4. ]


In [21]:
movie_enc.classes_[4]

"'Til There Was You (1997)"

In [22]:
test_user_id = 7

def extract_true_ratings(user_id_encoded, X_test):
    '''
    Extracts true ratings for a given ENCODED user ID from X_test.
    '''
    true_ratings = []
    # Iterate through X_test to find entries for the specific encoded user
    for x_encoded_user, x_encoded_movie in X_test:
        if x_encoded_user == user_id_encoded:
            # Get original user ID and movie title to query ratings_data
            original_user_id = user_enc.classes_[user_id_encoded]
            original_movie_title = movie_enc.classes_[x_encoded_movie]

            # Find the actual rating from the original ratings_data DataFrame
            rating_row = ratings_data[
                (ratings_data['userId'] == original_user_id) &
                (ratings_data['title'] == original_movie_title)
            ]
            if not rating_row.empty:
                true_ratings.append(rating_row['rating'].values[0])

    return true_ratings

# Encode the test_user_id before passing it to the function
encoded_test_user_id = user_enc.transform([test_user_id])[0]

true_ratings_for_test_user = extract_true_ratings(encoded_test_user_id, X_test)
print(f"True ratings for user {test_user_id}: {true_ratings_for_test_user}")

True ratings for user 7: [np.float64(3.5), np.float64(1.5), np.float64(3.5), np.float64(5.0), np.float64(4.0), np.float64(4.0), np.float64(4.0), np.float64(0.5), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(3.0), np.float64(1.0), np.float64(5.0), np.float64(4.5), np.float64(3.5), np.float64(4.0), np.float64(3.5), np.float64(0.5), np.float64(3.5), np.float64(4.5), np.float64(4.5), np.float64(0.5), np.float64(2.0), np.float64(1.5), np.float64(4.0), np.float64(3.0), np.float64(2.0), np.float64(3.0), np.float64(4.5), np.float64(2.5), np.float64(4.5), np.float64(5.0), np.float64(4.0), np.float64(4.0)]


In [23]:

def extract_true_ratings(user_id, X_test):

    true_ratings = list()
    for x, y in X_test:
        if x == user_id:
            rating = ratings_data[(ratings_data['userId'] == user_enc.classes_[user_id]) \
                & (ratings_data['title'] == movie_enc.classes_[y])]['rating'].values[0]
            true_ratings.append(rating)

    return true_ratings

In [24]:
def predict_ratings(user_id, X_test):
    '''
    given user id predict all ratings for movies
    '''
    user_data = ratings_data[ratings_data['userId'] == user_id]
    movie_ids, movie_names, predictions, movie_genres = list(), list(), list(), list()
    i = 0
    for _id, movie_id in X_test:
        if user_id == X_test[i][0]:
            movie_ids.append(X_test[i, 1])
            movie_names.append(movie_enc.classes_[movie_id])
            pred = model.predict([ np.array([X_test[i, 0]]), np.array([X_test[i, 1]]) ])
            predictions.append(pred[0][0])
        i += 1
    return movie_ids, movie_names, movie_genres, predictions

In [25]:

test_user_id = 7
userid_rating_data = ratings_data[ratings_data['userId'] == test_user_id]
# userid_rating_data

In [26]:
movie_ids, movie_names, movie_genres, predictions = predict_ratings(test_user_id, X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step


In [27]:
dictionary = {"user_id": [test_user_id]*len(movie_ids),
              "movie_id": movie_ids,
              "movie_name":movie_names,
              "predicted_ratings":predictions,
              "true_ratings": extract_true_ratings(test_user_id, X_test)
              }

In [28]:

prediction_dataframe = pd.DataFrame.from_dict(dictionary, orient='index').transpose()
prediction_dataframe.sort_values('predicted_ratings', ascending=False)

,user_id,movie_id,movie_name,predicted_ratings,true_ratings
3,7,1799,City Slickers II: The Legend of Curly's Gold (...,4.140036,1.0
5,7,7421,Schindler's List (1993),3.852367,5.0
4,7,2139,Dances with Wolves (1990),3.400896,5.0
1,7,7755,Sleepless in Seattle (1993),3.120554,3.0
0,7,6865,Pulp Fiction (1994),3.073096,4.0
2,7,7912,Speed (1994),2.93403,4.0
